In [7]:
import pandas as pd
URL = "https://raw.githubusercontent.com/tiwvis/MLBD-Projects/main/data_sets/LoansData_sample.csv.gz"
df = pd.read_csv(URL, compression="gzip", low_memory=False)
print(df.shape)
print(df["loan_status"].value_counts())
print([c for c in df.columns if "date" in c.lower() or c.endswith("_d")])

(100000, 151)
loan_status
Fully Paid            69982
Charged Off           16156
Current               12726
Late (31-120 days)      730
In Grace Period         264
Late (16-30 days)       139
Default                   3
Name: count, dtype: int64
['issue_d', 'last_pymnt_d', 'next_pymnt_d', 'last_credit_pull_d', 'hardship_start_date', 'hardship_end_date', 'payment_plan_start_date', 'debt_settlement_flag_date', 'settlement_date']


In [8]:
d = pd.to_datetime(df["issue_d"], format="%b-%Y", errors="coerce")
lab = df["loan_status"].isin(["Fully Paid", "Charged Off", "Default"])
tmp = pd.DataFrame({"year": d.dt.year,
                    "default": df["loan_status"].isin(["Charged Off", "Default"]).astype(int)})[lab]
print(tmp.groupby("year")["default"].agg(["count", "mean"]))
print("unparseable dates:", d.isna().sum())

      count      mean
year                 
2014  86141  0.187588
unparseable dates: 0


In [9]:
import numpy as np

# Target: keep only loans with a final outcome
df = df[df["loan_status"].isin(["Fully Paid", "Charged Off", "Default"])].copy()
df["default"] = df["loan_status"].isin(["Charged Off", "Default"]).astype(int)

# Drop leakage columns (post-issuance info), identifiers and free text
DROP = ["out_prncp","out_prncp_inv","total_pymnt","total_pymnt_inv","total_rec_prncp",
        "total_rec_int","total_rec_late_fee","recoveries","collection_recovery_fee",
        "last_pymnt_d","last_pymnt_amnt","next_pymnt_d","last_credit_pull_d",
        "last_fico_range_high","last_fico_range_low","debt_settlement_flag","policy_code",
        "pymnt_plan","loan_status","id","member_id","url","desc","title","emp_title",
        "zip_code","funded_amnt","funded_amnt_inv"]
DROP += [c for c in df.columns if c.startswith(("hardship_","settlement_","sec_app_","orig_projected","deferral","payment_plan"))]
df = df.drop(columns=[c for c in DROP if c in df.columns])

# Dates -> credit history length (keep issue_dt only for the split)
df["issue_dt"] = pd.to_datetime(df["issue_d"], format="%b-%Y", errors="coerce")
earliest = pd.to_datetime(df["earliest_cr_line"], format="%b-%Y", errors="coerce")
df["credit_hist_years"] = (df["issue_dt"] - earliest).dt.days / 365.25
df = df.drop(columns=["issue_d", "earliest_cr_line"])

# Term and employment length
df["term_months"] = pd.to_numeric(df["term"].astype(str).str.extract(r"(\d+)")[0])
el = df["emp_length"].astype(str)
df["emp_length_yrs"] = np.where(el.str.contains(r"10\+"), 10,
                       np.where(el.str.contains("< 1"), 0,
                                pd.to_numeric(el.str.extract(r"(\d+)")[0], errors="coerce")))
df = df.drop(columns=["term", "emp_length"])

# Remaining text columns: categoricals stay, everything else becomes numeric
CATEGORICAL = ["grade","sub_grade","home_ownership","verification_status","purpose",
               "addr_state","initial_list_status","application_type","disbursement_method"]
CATEGORICAL = [c for c in CATEGORICAL if c in df.columns]
for c in df.select_dtypes("object").columns:
    if c not in CATEGORICAL:
        df[c] = pd.to_numeric(df[c].astype(str).str.replace("%", "", regex=False), errors="coerce")

# Checks
print(df.shape, "| default rate:", round(df["default"].mean(), 4))
print("object columns left:", df.select_dtypes("object").columns.tolist())
print("columns with >50% missing:", df.columns[df.isna().mean() > 0.5].tolist())

(86141, 93) | default rate: 0.1876
object columns left: ['grade', 'sub_grade', 'home_ownership', 'verification_status', 'purpose', 'addr_state', 'initial_list_status', 'application_type', 'disbursement_method']
columns with >50% missing: ['mths_since_last_record', 'mths_since_last_major_derog', 'annual_inc_joint', 'dti_joint', 'verification_status_joint', 'open_acc_6m', 'open_act_il', 'open_il_12m', 'open_il_24m', 'mths_since_rcnt_il', 'total_bal_il', 'il_util', 'open_rv_12m', 'open_rv_24m', 'max_bal_bc', 'all_util', 'inq_fi', 'total_cu_tl', 'inq_last_12m', 'mths_since_recent_bc_dlq', 'mths_since_recent_revol_delinq', 'revol_bal_joint', 'debt_settlement_flag_date']


In [10]:
USE_TIME_SPLIT = False   # set False for a stratified random split

if USE_TIME_SPLIT:
    q70, q85 = df["issue_dt"].quantile([0.70, 0.85])
    df["split"] = np.where(df["issue_dt"] <= q70, "train",
                  np.where(df["issue_dt"] <= q85, "val", "test"))
else:
    from sklearn.model_selection import train_test_split
    idx_tr, idx_rest = train_test_split(df.index, test_size=0.30, stratify=df["default"], random_state=42)
    idx_va, idx_te = train_test_split(idx_rest, test_size=0.50, stratify=df.loc[idx_rest, "default"], random_state=42)
    df["split"] = "train"
    df.loc[idx_va, "split"] = "val"
    df.loc[idx_te, "split"] = "test"

print(df.groupby("split").agg(n=("default", "size"), default_rate=("default", "mean"),
                              first=("issue_dt", "min"), last=("issue_dt", "max")))

# Drop columns that are >50% missing, judged on the training rows only
miss = df.loc[df["split"] == "train"].isna().mean()
drop_cols = miss[miss > 0.5].index.tolist()
df = df.drop(columns=drop_cols)
print("dropped for missingness:", len(drop_cols), "| columns left:", df.shape[1])

           n  default_rate      first       last
split                                           
test   12922      0.187587 2014-08-01 2014-12-01
train  60298      0.187585 2014-08-01 2014-12-01
val    12921      0.187602 2014-08-01 2014-12-01
dropped for missingness: 23 | columns left: 71


In [11]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.makedirs("/content/drive/MyDrive/loan-default-mlbd", exist_ok=True)
OUT = "/content/drive/MyDrive/loan-default-mlbd/loans_clean.parquet"
df.to_parquet(OUT, index=False)
print(pd.read_parquet(OUT).shape)   # reload check

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
(86141, 71)


In [12]:
import sys; sys.path.append("/content/loan-default-mlbd")   # or wherever the repo is cloned in Colab